In [9]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from IPython.display import Image, display

import gc
import itertools


In [2]:
path = "~/Downloads/AO2Dtree.root"
file = uproot.open(path)

In [3]:
# NSigmaTPC:
sigma_limit = 3
cut1 = (
    "( (fNsigmaTPCpi > -3) & (fNsigmaTPCpi < 3) & (fCharge == 1) ) | "
    "( (fNsigmaTPCka > -3) & (fNsigmaTPCka < 3) & (fCharge == -1) ) | "
    "( (fPt > 0) & (fPt < 1) & (fNsigmaTPCka > -15) & (fNsigmaTPCka < 0) & (fCharge == 1) )"
)

# NsigmaTOF:
sigma_limit = 3
cut2 = (
    "( (fNsigmaTOFpi > -3) & (fNsigmaTOFpi < 3) & (fCharge == 1) ) | "
    "( (fNsigmaTOFka > -3) & (fNsigmaTOFka < 3) & (fCharge == -1) ) | "
    "( (fNsigmaTOFka > 998.5) | (fNsigmaTOFka < -998.5) | (fNsigmaTOFpi > 998.5) | (fNsigmaTOFpi < -998.5) ) | "
    "( (fPt > 0) & (fPt < 3) & (fNsigmaTOFka > -50) & (fNsigmaTOFka < 0) & (fCharge == 1) )"
)

# DCA_XY:
cut3 = "( (fDcaXY > 0.0002) | (fDcaXY < -0.0002) )"


# FINAL CUT EXPRESSION:
cut_expression = f"({cut1}) & ({cut2}) & ({cut3})"

In [18]:
# CODE FOR UNIFYING DIRECTORIES OF A SINGLE ROOT FILE from all three type of T-Trees:

# Load T-Trees from directories
names_track_extr = file.keys(filter_name=r"*O2filtertrackextr")
names_track      = file.keys(filter_name=r"*O2filtertrack")
names_coll       = file.keys(filter_name=r"*O2collision_001")
#num = 400   # number of directories to unify
num = len(names_coll)
# if I try with all, it crashes, because in this notebook I'm doing the cuts only after loading all data (in order tp study the cuts)

list_of_df = []               # add the dataframes in a list (we will concat them later)

collision_offset = 0          # variable to correct the index of the collision (because it starts from 0 at each new TTree)

for i in range(num):          # cycle over directories (each with 3 TTrees)
    
    # Read collision tree
    df_coll = file[ names_coll[i] ].arrays(["fPosZ"], library="pd")   # I take the fPosZ column as a DataFrame
    # list_of_collision_df.append(df_coll)               # add the dataframe in a list (we will concat them later)

    # # Read track and trackextr using boolean mask for track and trackextr:
    df_trackextr = file[ names_track_extr[i] ].arrays(["fPt", "fEta", "fCharge", "fDcaXY",
         "fNsigmaTPCpi", "fNsigmaTPCka", "fNsigmaTPCpr", "fNsigmaTOFpi", "fNsigmaTOFka", "fNsigmaTOFpr"], library="pd" )
    df_track = file[ names_track[i] ].arrays(["fIndexCollisions", "fAlpha", "fX", "fY", "fZ"], library="pd")
    mask = df_trackextr.eval(cut_expression)        # create boolean mask
    # Apply the SAME filter to df_trackextr and df_track to keep them aligned (and keep only useful columns):
    df_trackextr = df_trackextr.loc[mask, ["fPt", "fEta", "fCharge", "fDcaXY"] ].reset_index(drop=True)
    df_track = df_track.loc[mask,].reset_index(drop=True)
    # merging in a single dataframe
    df_trackextr["fIndexCollisions"] = df_track["fIndexCollisions"] 
    df_trackextr["fAlpha"] = df_track["fAlpha"]
    df_trackextr["fX"] = df_track["fX"]
    df_trackextr["fY"] = df_track["fY"]
    df_trackextr["fZ"] = df_track["fZ"]

    # we cut rows where the fIndexCollision is negative (for some reason)
    valid = df_track["fIndexCollisions"] >= 0
    df_track = df_track[valid].reset_index(drop=True)          
    df_trackextr = df_trackextr[valid].reset_index(drop=True)  

    
    # Now we for correct fPosZ (and add that column)
    df_trackextr["fPosZ"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosZ"].values
    df_trackextr = df_trackextr[(df_trackextr["fPosZ"] < 10) & (df_trackextr["fPosZ"] > -10)].reset_index(drop=True)
    df_trackextr["fIndexCollisions"] += collision_offset   # Fix local fIndexCollisions → global index 

    # save results:
    # ALTERNATIVE 1: for the first cycle, let's copy the first dataframe, then we concatenate the next ones
    if  i==0: df = df_trackextr
    else: df = pd.concat([df, df_trackextr], ignore_index=True)
    # # ALTERNATIVE 2:
    # list_of_df.append( df_trackextr )                  # add the dataframe in a list (we will concat them later)
    
    # Update offset for next loop
    collision_offset += len(df_coll)     

    # let's free the memory RAM of unused dataframes:
    del df_trackextr
    del df_track
    gc.collect()


# # UNCOMMENT FOR ALTERNATIVE 2:
# df = pd.concat(list_of_df, ignore_index=True)

# Merge everything in the total dataframe
N = len(df)


In [19]:
# debug
print("The dataframe has", len(df), "rows")
memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"The dataframe occupy {memory:.2f} MB")
df

The dataframe has 4878200 rows
The dataframe occupy 186.09 MB


,fPt,fEta,fCharge,fDcaXY,fIndexCollisions,fAlpha,fX,fY,fZ,fPosZ
0,0.663068,0.598317,1,0.004326,0,-0.322267,-0.017226,-0.030717,-4.129672,-4.128410
1,1.434872,0.156435,-1,0.001984,2,-0.783967,-0.007016,-0.037420,-8.385111,-8.386139
2,1.418317,-0.302028,-1,-0.000906,3,1.478421,-0.035237,0.028537,4.196383,4.198601
3,0.629268,-0.664176,-1,-0.001437,3,1.949002,-0.018057,0.040782,4.195937,4.198601
4,1.047158,-0.719631,1,-0.006082,3,-0.223422,-0.024587,-0.044864,4.194757,4.198601
...,...,...,...,...,...,...,...,...,...,...
4878195,0.521796,0.090340,-1,0.004239,1666064,-1.776574,0.034615,-0.015266,-6.391246,-6.388824
4878196,0.924928,-0.743287,-1,-0.008164,1666064,2.492877,0.002787,0.031470,-6.380364,-6.388824
4878197,0.638627,-0.544448,-1,-0.000368,1666065,-2.320294,0.047770,-0.004037,-4.530087,-4.533272
4878198,0.915340,0.540868,1,-0.005042,1666065,1.709269,-0.027295,0.034333,-4.541160,-4.533272


In [20]:
# Particle masses in GeV
m_K = 0.493677
m_pi = 0.139570
m_d0 = 1.86484

# moment columns:
df["px"] = df["fPt"] * np.cos(df["fAlpha"])
df["py"] = df["fPt"] * np.sin(df["fAlpha"])
df["pz"] = df["fPt"] * np.sinh(df["fEta"])

# energy column (differentiating pions and kaons):
mass = np.where(df["fCharge"] > 0, m_pi, m_K)
df["Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + mass**2)

# Debug
print("The dataframe has", len(df), "rows")
memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"The dataframe occupy {memory:.2f} MB")
df

The dataframe has 4878200 rows
The dataframe occupy 279.13 MB


,fPt,fEta,fCharge,fDcaXY,fIndexCollisions,fAlpha,fX,fY,fZ,fPosZ,px,py,pz,Ene
0,0.663068,0.598317,1,0.004326,0,-0.322267,-0.017226,-0.030717,-4.129672,-4.128410,0.628933,-0.210006,0.420822,0.797640
1,1.434872,0.156435,-1,0.001984,2,-0.783967,-0.007016,-0.037420,-8.385111,-8.386139,1.016059,-1.013154,0.225381,1.534070
2,1.418317,-0.302028,-1,-0.000906,3,1.478421,-0.035237,0.028537,4.196383,4.198601,0.130831,1.412270,-0.434913,1.563487
3,0.629268,-0.664176,-1,-0.001437,3,1.949002,-0.018057,0.040782,4.195937,4.198601,-0.232359,0.584797,-0.449357,0.917397
4,1.047158,-0.719631,1,-0.006082,3,-0.223422,-0.024587,-0.044864,4.194757,4.198601,1.021130,-0.232017,-0.820313,1.337510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4878195,0.521796,0.090340,-1,0.004239,1666064,-1.776574,0.034615,-0.015266,-6.391246,-6.388824,-0.106618,-0.510787,0.047203,0.719872
4878196,0.924928,-0.743287,-1,-0.008164,1666064,2.492877,0.002787,0.031470,-6.380364,-6.388824,-0.737039,0.558808,-0.752563,1.290566
4878197,0.638627,-0.544448,-1,-0.000368,1666065,-2.320294,0.047770,-0.004037,-4.530087,-4.533272,-0.435078,-0.467495,-0.365133,0.885936
4878198,0.915340,0.540868,1,-0.005042,1666065,1.709269,-0.027295,0.034333,-4.541160,-4.533272,-0.126345,0.906579,0.519572,1.061735


In [28]:
def to_global_coord( row ):
    xy_in = np.array([row["fX"],row["fY"]])
    angle = +row["fAlpha"]
    rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])  # classical rotation matrix
    xy_out = rot.dot(xy_in)
    return( xy_out )

def secondary_vertex ( row1, row2 ):
    XY1, XY2 = [to_global_coord(row1), to_global_coord(row2)]
    x1 = XY1[0]
    y1 = XY1[1]
    x2 = XY2[0]
    y2 = XY2[1]
    
    px1 = row1["px"]
    px2 = row2["px"]
    py1 = row1["py"]
    py2 = row2["py"]
    pz1 = row1["pz"]
    pz2 = row2["pz"]
    
    m1 = py1/px1
    m2 = py2/px2
    q1 = y1 - m1*x1
    q2 = y2 - m2*x2
    x_SV = ( q2 - q1 )/( m1 - m2)
    y_SV = y1 + m1*(x_SV - x1)
   
    z1_track = pz1/px1 * x_SV + row1["fZ"]
    z2_track = pz2/px2 * x_SV + row2["fZ"]
    z_SV = (z1_track + z2_track)/2
    return ([x_SV, y_SV, z_SV])

In [29]:
# ALTERNATIVE 3:
# let's initialize some lists, then we will create a dataframe
collision_indices = []
track1_indices = []
track2_indices = []
dcaXY_products = []
inv_masses = []
inv_masses_approx = []
pt_totals = []
SV_X = []
SV_Y = []
SV_Z = []

i=0 # debug variable

# let's divide the dataframe for positive and negative charged
df_pos = df[ df['fCharge']>0 ]
df_neg = df[ df['fCharge']<0 ]

# ATTENTION: Introduced to run reasonably fast, remember to remove when running over all data on CloudVeneto
max_iters = 10_001

# Iterate over each collision group
for collision_idx in (df_neg['fIndexCollisions'].unique())[:max_iters]:
    group_pos = df_pos[ df_pos['fIndexCollisions'] == collision_idx ]
    group_neg = df_neg[ df_neg['fIndexCollisions'] == collision_idx ]

    # Only collisions with at least a pair
    if len(group_pos) < 1:   continue

    # Reset index of the group to 0..N-1 and move original index in new column 'orig_index'
    group_pos = group_pos.reset_index().rename(columns={'index': 'orig_index'})
    group_neg = group_neg.reset_index().rename(columns={'index': 'orig_index'})

    # let's crate indexes for all possible pairs:
    combinat = itertools.product( range(len(group_neg)), range(len(group_pos)) )

    # Iterate over all unique pairs of tracks
    for combo in combinat:
        row_neg = group_neg.iloc[combo[0]]
        row_pos = group_pos.iloc[combo[1]]

        product_dcaXY = row_neg['fDcaXY'] * row_pos['fDcaXY']
        
        # INVARIANT MASS calculation
        pt1, pt2 = row_neg['fPt'], row_pos['fPt']
        eta1, eta2 = row_neg['fEta'], row_pos['fEta']
        phi1, phi2 = row_neg['fAlpha'], row_pos['fAlpha']
        delta_eta = eta1 - eta2
        delta_phi = phi1 - phi2

        # approximation formula
        inv_mass_approx = np.sqrt(2 * pt1 * pt2 * (np.cosh(delta_eta) - np.cos(delta_phi)))

        # exact formula:
        E1 = row_neg['Ene']
        E2 = row_pos['Ene']
        px1 = row_neg["px"]
        py1 = row_neg["py"]
        pz1 = row_neg["pz"]
        px2 = row_pos["px"]
        py2 = row_pos["py"]
        pz2 = row_pos["pz"]
        inv_mass = np.sqrt( (E1+E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )


        # total transverse momentum of the D0 candidate (used later for sliced plots)
        pt_total = np.sqrt((px1 + px2)**2 + (py1 + py2)**2)

        # secondary vertex
        global_x_sv, global_y_sv, global_z_sv = secondary_vertex(row_neg, row_pos)
        SV_X.append(global_x_sv)
        SV_Y.append(global_y_sv)
        SV_Z.append(global_z_sv)
        
        # let's add the found pairs to the lists
        collision_indices.append(int(row_neg['fIndexCollisions']))
        track1_indices.append(int(row_neg['orig_index']))
        track2_indices.append(int(row_pos['orig_index']))
        dcaXY_products.append(product_dcaXY)
        inv_masses.append(inv_mass)
        inv_masses_approx.append(inv_mass_approx)
        pt_totals.append(pt_total)
        SV.append(sec_vertex)

    # # let's free the memory RAM of unused dataframes:
    # # PROBLEM: THIS IS VERY SLOW!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    # del group
    # gc.collect()

    # debug:
    i += 1
    if i % 10000 ==0: print(i, end=' ')


# create a dataframe with the result:
df_pairs = pd.DataFrame({
    'collision_index': collision_indices,
    'track1_index': track1_indices,
    'track2_index': track2_indices,
    'dcaXY_product': dcaXY_products,
    'inv_mass': inv_masses,
    'inv_mass_approx': inv_masses_approx,
    'pt': pt_totals,
    'X_SV': SV_X,
    'Y_SV': SV_Y,
    'Z_SV': SV_Z
})

In [30]:
# Debug
print("The dataframe has", len(df_pairs), "rows")
memory = df_pairs.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"The dataframe occupy {memory:.2f} MB")
df_pairs

The dataframe has 80147 rows
The dataframe occupy 6.11 MB


,collision_index,track1_index,track2_index,dcaXY_product,inv_mass,inv_mass_approx,pt,X_SV,Y_SV,Z_SV
0,3,2,4,5.512908e-06,2.029822,1.902984,1.649246,-0.032243,-0.038682,4.262112
1,3,2,5,-8.264114e-07,1.177901,0.930195,2.772398,-0.031399,-0.029569,4.251124
2,3,2,6,7.114767e-06,0.837150,0.547550,1.725034,-0.032643,-0.043004,4.260567
3,3,2,7,-5.188740e-07,2.151625,2.048243,0.695278,-0.031325,-0.028772,4.199664
4,3,2,8,4.383081e-06,1.363588,1.120530,3.236413,-0.032280,-0.039090,4.257296
...,...,...,...,...,...,...,...,...,...,...
80142,25291,72720,72726,-2.352030e-05,0.843383,0.555957,1.518973,-0.028974,-0.016276,-0.286866
80143,25299,72738,72737,-2.083822e-06,1.448382,1.267970,0.906922,-0.030565,-0.026354,-0.968269
80144,25299,72738,72740,6.520136e-07,1.479234,1.320576,0.309748,-0.029826,-0.016710,-0.926444
80145,25299,72739,72737,-2.961663e-06,1.597018,1.458194,1.565676,-0.028996,-0.025948,-0.941434
